In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Locate the project root
current = Path.cwd()

project_root = next(
    (
        p for p in [current, *current.parents]
        if (p / "data" / "processed" / "perishable_inventory.csv").exists()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Cannot find data/processed/perishable_inventory.csv. "
        "Open the notebook inside the WasteWise-AI project."
    )

DATA_PATH = project_root / "data" / "processed" / "perishable_inventory.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Dataset path:", DATA_PATH)
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Dataset loaded successfully!
Dataset path: c:\Users\hp\OneDrive\Desktop\web_scraping_folder\wasteAI_project\WasteWise-AI\data\processed\perishable_inventory.csv
Rows: 1800
Columns: 12


In [2]:
# Step 2: Understand the dataset structure

print("DATASET SHAPE")
print("-" * 40)
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

print("\nCOLUMN NAMES")
print("-" * 40)
print(df.columns.tolist())

print("\nDATA TYPES AND NON-NULL COUNTS")
print("-" * 40)
df.info()

print("\nMISSING VALUES")
print("-" * 40)
missing_values = df.isnull().sum()
print(missing_values)

print("\nDUPLICATE ROWS")
print("-" * 40)
print("Duplicate rows:", df.duplicated().sum())

print("\nNUMERICAL SUMMARY")
print("-" * 40)
display(df.describe().round(2))

print("\nFIRST 5 ROWS")
print("-" * 40)
display(df.head())

DATASET SHAPE
----------------------------------------
Rows: 1800
Columns: 12

COLUMN NAMES
----------------------------------------
['product_id', 'date', 'category', 'opening_stock', 'units_received', 'units_sold', 'unit_price', 'discount_pct', 'days_to_expiry', 'temperature_c', 'waste_units', 'stockout_flag']

DATA TYPES AND NON-NULL COUNTS
----------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 1800 entries, 0 to 1799
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   product_id      1800 non-null   int64  
 1   date            1800 non-null   str    
 2   category        1800 non-null   str    
 3   opening_stock   1800 non-null   int64  
 4   units_received  1800 non-null   int64  
 5   units_sold      1800 non-null   int64  
 6   unit_price      1800 non-null   float64
 7   discount_pct    1800 non-null   int64  
 8   days_to_expiry  1800 non-null   int64  
 9   temperature_c   1

,product_id,opening_stock,units_received,units_sold,unit_price,discount_pct,days_to_expiry,temperature_c,waste_units,stockout_flag
count,1800.00,1800.00,1800.00,1800.00,1800.00,1800.00,1800.00,1800.00,1800.00,1800.00
mean,1005.50,35.36,2.42,20.15,57.86,1.34,4.67,14.74,1.45,0.08
std,2.87,13.50,5.03,9.89,29.63,4.28,5.36,8.97,5.98,0.26
min,1001.00,0.00,0.00,0.00,20.00,0.00,0.00,-0.40,0.00,0.00
25%,1003.00,26.00,0.00,12.00,32.00,0.00,1.00,4.50,0.00,0.00
50%,1005.50,35.00,0.00,19.00,45.00,0.00,3.00,20.50,0.00,0.00
75%,1008.00,45.00,0.00,27.00,85.00,0.00,6.00,22.20,0.00,0.00
max,1010.00,80.00,26.00,53.00,120.00,20.00,30.00,26.70,56.00,1.00



FIRST 5 ROWS
----------------------------------------


,product_id,date,category,opening_stock,units_received,units_sold,unit_price,discount_pct,days_to_expiry,temperature_c,waste_units,stockout_flag
0,1001,2026-04-01,Dairy,27,0,6,32.0,0,1,4.1,0,0
1,1001,2026-04-02,Dairy,30,12,5,32.0,0,5,3.7,0,0
2,1001,2026-04-03,Dairy,38,0,14,32.0,0,2,4.6,0,0
3,1001,2026-04-04,Dairy,49,0,11,32.0,0,0,5.7,0,0
4,1001,2026-04-05,Dairy,19,13,12,32.0,0,0,3.0,0,0


In [3]:
# Step 3: Understand dates, products, and categories

# Convert date column to datetime format
df["date"] = pd.to_datetime(df["date"], errors="coerce")

print("DATE RANGE")
print("-" * 40)
print("First date:", df["date"].min())
print("Latest date:", df["date"].max())
print("Unique dates:", df["date"].nunique())
print("Invalid dates:", df["date"].isna().sum())

print("\nPRODUCT ANALYSIS")
print("-" * 40)
print("Unique products:", df["product_id"].nunique())
display(
    df.groupby("product_id")
    .agg(
        records=("date", "count"),
        total_units_sold=("units_sold", "sum"),
        total_waste=("waste_units", "sum"),
    )
    .reset_index()
)

print("\nCATEGORY DISTRIBUTION")
print("-" * 40)
display(df["category"].value_counts().rename_axis("category").reset_index(name="records"))

print("\nSALES SUMMARY")
print("-" * 40)
print("Total units sold:", df["units_sold"].sum())
print("Average units sold per record:", round(df["units_sold"].mean(), 2))
print("Total recorded waste units:", df["waste_units"].sum())

print("\nSTOCKOUT SUMMARY")
print("-" * 40)
display(df["stockout_flag"].value_counts(dropna=False))

DATE RANGE
----------------------------------------
First date: 2026-04-01 00:00:00
Latest date: 2026-09-27 00:00:00
Unique dates: 180
Invalid dates: 0

PRODUCT ANALYSIS
----------------------------------------
Unique products: 10


,product_id,records,total_units_sold,total_waste
0,1001,180,1868,313
1,1002,180,5408,311
2,1003,180,4672,195
3,1004,180,3535,275
4,1005,180,3525,383
5,1006,180,6001,189
6,1007,180,1902,238
7,1008,180,5015,277
8,1009,180,2428,157
9,1010,180,1913,264



CATEGORY DISTRIBUTION
----------------------------------------


,category,records
0,Dairy,720
1,Produce,540
2,Bakery,360
3,Beverage,180



SALES SUMMARY
----------------------------------------
Total units sold: 36267
Average units sold per record: 20.15
Total recorded waste units: 2602

STOCKOUT SUMMARY
----------------------------------------


stockout_flag
0    1664
1     136
Name: count, dtype: int64

In [5]:
# Step 3: Date and Product Analysis

df["date"] = pd.to_datetime(df["date"], errors="coerce")

print("DATE RANGE")
print("-" * 40)
print("First date:", df["date"].min())
print("Latest date:", df["date"].max())
print("Unique dates:", df["date"].nunique())
print("Invalid dates:", df["date"].isna().sum())

print("\nPRODUCT ANALYSIS")
print("-" * 40)
print("Unique products:", df["product_id"].nunique())

product_summary = (
    df.groupby("product_id")
    .agg(
        records=("date", "count"),
        total_units_sold=("units_sold", "sum"),
        total_waste=("waste_units", "sum"),
    )
    .reset_index()
)

display(product_summary)

print("\nCATEGORY DISTRIBUTION")
print("-" * 40)
display(df["category"].value_counts())

print("\nSALES AND WASTE SUMMARY")
print("-" * 40)
print("Total units sold:", df["units_sold"].sum())
print("Average sales per record:", round(df["units_sold"].mean(), 2))
print("Total waste units:", df["waste_units"].sum())

print("\nSTOCKOUT SUMMARY")
print("-" * 40)
display(df["stockout_flag"].value_counts(dropna=False))

DATE RANGE
----------------------------------------
First date: 2026-04-01 00:00:00
Latest date: 2026-09-27 00:00:00
Unique dates: 180
Invalid dates: 0

PRODUCT ANALYSIS
----------------------------------------
Unique products: 10


,product_id,records,total_units_sold,total_waste
0,1001,180,1868,313
1,1002,180,5408,311
2,1003,180,4672,195
3,1004,180,3535,275
4,1005,180,3525,383
5,1006,180,6001,189
6,1007,180,1902,238
7,1008,180,5015,277
8,1009,180,2428,157
9,1010,180,1913,264



CATEGORY DISTRIBUTION
----------------------------------------


category
Dairy       720
Produce     540
Bakery      360
Beverage    180
Name: count, dtype: int64


SALES AND WASTE SUMMARY
----------------------------------------
Total units sold: 36267
Average sales per record: 20.15
Total waste units: 2602

STOCKOUT SUMMARY
----------------------------------------


stockout_flag
0    1664
1     136
Name: count, dtype: int64

In [4]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Locate the WasteWise AI project root
current = Path.cwd()

project_root = next(
    (
        path
        for path in [current, *current.parents]
        if (
            path / "data" / "processed"
            / "perishable_inventory.csv"
        ).exists()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Could not locate the processed sales dataset."
    )

DATA_PATH = (
    project_root
    / "data"
    / "processed"
    / "perishable_inventory.csv"
)

df = pd.read_csv(DATA_PATH)
df["date"] = pd.to_datetime(df["date"], errors="coerce")

print("EDA dataset loaded successfully!")
print("Dataset shape:", df.shape)
print("Date range:", df["date"].min(), "to", df["date"].max())
print("Unique products:", df["product_id"].nunique())
print("Categories:", df["category"].nunique())

display(df.head())

EDA dataset loaded successfully!
Dataset shape: (1800, 12)
Date range: 2026-04-01 00:00:00 to 2026-09-27 00:00:00
Unique products: 10
Categories: 4


,product_id,date,category,opening_stock,units_received,units_sold,unit_price,discount_pct,days_to_expiry,temperature_c,waste_units,stockout_flag
0,1001,2026-04-01,Dairy,27,0,6,32.0,0,1,4.1,0,0
1,1001,2026-04-02,Dairy,30,12,5,32.0,0,5,3.7,0,0
2,1001,2026-04-03,Dairy,38,0,14,32.0,0,2,4.6,0,0
3,1001,2026-04-04,Dairy,49,0,11,32.0,0,0,5.7,0,0
4,1001,2026-04-05,Dairy,19,13,12,32.0,0,0,3.0,0,0
